In [3]:
%mamba install pandas

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 1.834 seconds
  Name                          Version                       Build                         Channel                       
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
+ pandas                        3.0.1                         np22py313h9d9dc1e_0           emscripten-forge              
+ python-tzdata                 2025.3                        pyhd8ed1ab_0                  conda-forge                   
- pip                           25.3                          pyh145f28c_0                  conda-forge                   


In [4]:
import numpy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

In [5]:
# Import CAISO LMP data for 2025-12-10
df_lmp = pd.read_csv("1_day_LMP_DAM.csv")
# Import CAISO actual load data
df_actual = pd.read_csv("1_month_CAISO_load.csv")
# Import CAISO DAM (Day ahead market) forecast 
df_dam = pd.read_csv("1_month_DAM_forecast.csv")
# Import CAISO RTM (Real time market) forecast
df_rtm = pd.read_csv("1_month_RTM_forecast.csv")

In [6]:
# Define a function to parse datetime fields and set index
def parse_dates(df): 
    df['start_time'] = pd.to_datetime(df['INTERVALSTARTTIME_GMT'], utc=True, errors='coerce')
    df['end_time'] = pd.to_datetime(df['INTERVALENDTIME_GMT'], utc=True, errors='coerce')

    df = df.set_index('start_time').sort_index()
    print("Missing indexes:" + str(df.index.isna().sum())) 

    df['day'] = df.index.day
    df['weekday'] = df.index.weekday
    df['day_of_week'] = df.index.strftime('%a')

    return df

In [7]:
def time_series_checks(df):
    # Confirm frequency consistency
    print("Index frequency: "+ str(pd.infer_freq(df.index)))
    
    # Check for duplicate timestamps
    print("Duplicate timestamps: "+ str(df.index.duplicated().sum()))
    
    # Find sequences of consecutive NaN values and count how long each sequence is
    result = df['MW'].isna().astype(int).groupby(
        df['MW'].notna().cumsum()
    ).sum()
    print("Sequences of consecutive NaN values: ")
    print(result[result > 0])
    print("\n")

In [8]:
df_lmp = parse_dates(df_lmp)
time_series_checks(df_lmp)

Missing indexes:0
Index frequency: None
Duplicate timestamps: 88480
Sequences of consecutive NaN values: 
Series([], Name: MW, dtype: int32)




In [9]:
print(df_lmp.describe())
print(df_lmp.info())

             OPR_HR  OPR_INTERVAL      POS            MW         GROUP  \
count  88504.000000       88504.0  88504.0  88504.000000  88504.000000   
mean      12.499808           0.0      0.0     12.954888   1844.295851   
std        6.922147           0.0      0.0     20.245210   1064.537226   
min        1.000000           0.0      0.0    -18.142990      1.000000   
25%        6.750000           0.0      0.0      0.000000    922.000000   
50%       12.000000           0.0      0.0      0.000000   1844.000000   
75%       18.000000           0.0      0.0     35.813280   2766.000000   
max       24.000000           0.0      0.0     68.946170   3688.000000   

                day       weekday  
count  88504.000000  88504.000000  
mean      10.333307      2.333307  
std        0.471398      0.471398  
min       10.000000      2.000000  
25%       10.000000      2.000000  
50%       10.000000      2.000000  
75%       11.000000      3.000000  
max       11.000000      3.000000  
<class 'p

In [10]:
print(df_lmp['LMP_TYPE'].unique())

<StringArray>
['MCE', 'MGHG', 'MCL', 'MCC', 'LMP']
Length: 5, dtype: str


In [11]:
"""
| Component | Meaning                                 |
| --------- | --------------------------------------- |
| **MCE**   | Marginal Cost of Energy                 |
| **MCC**   | Marginal Cost of Congestion             |
| **MCL**   | Marginal Cost of Losses                 |
| **MGHG**  | Marginal greenhouse gas compliance cost |
"""

'\n| Component | Meaning                                 |\n| --------- | --------------------------------------- |\n| **MCE**   | Marginal Cost of Energy                 |\n| **MCC**   | Marginal Cost of Congestion             |\n| **MCL**   | Marginal Cost of Losses                 |\n| **MGHG**  | Marginal greenhouse gas compliance cost |\n'

In [13]:
"""
for lmp_type, n in df_lmp.groupby('LMP_TYPE'):
    if lmp_type =='MCE': df_MCE = n
    elif lmp_type == 'MCC': df_MCC = n
    elif lmp_type == 'MCL': df_MCL = n
    elif lmp_type == 'MGHG': df_MGHG = n
    else: df_only_lmp = n

print(df_MCE)
df_MCE.rename(columns={"MW": "MW_MCE"}, inplace=True)
df_MCC.rename(columns={"MW": "MW_MCC"}, inplace=True)
df_MCL.rename(columns={"MW": "MW_MCL"}, inplace=True)
df_MGHG.rename(columns={"MW": "MW_MGHG"}, inplace=True)


df_merged = pd.merge(df_MCE, df_MCC, on=['start_time','TAC_AREA_NAME'], how='inner')
df_merged = pd.merge(df_merged, df3, on=['start_time','TAC_AREA_NAME'], how='inner')
df_merged = pd.merge(df_merged, df4, on=['start_time','TAC_AREA_NAME'], how='inner')
"""

'\nfor lmp_type, n in df_lmp.groupby(\'LMP_TYPE\'):\n    if lmp_type ==\'MCE\': df_MCE = n\n    elif lmp_type == \'MCC\': df_MCC = n\n    elif lmp_type == \'MCL\': df_MCL = n\n    elif lmp_type == \'MGHG\': df_MGHG = n\n    else: df_only_lmp = n\n\nprint(df_MCE)\ndf_MCE.rename(columns={"MW": "MW_MCE"}, inplace=True)\ndf_MCC.rename(columns={"MW": "MW_MCC"}, inplace=True)\ndf_MCL.rename(columns={"MW": "MW_MCL"}, inplace=True)\ndf_MGHG.rename(columns={"MW": "MW_MGHG"}, inplace=True)\n\n\ndf_merged = pd.merge(df_MCE, df_MCC, on=[\'start_time\',\'TAC_AREA_NAME\'], how=\'inner\')\ndf_merged = pd.merge(df_merged, df3, on=[\'start_time\',\'TAC_AREA_NAME\'], how=\'inner\')\ndf_merged = pd.merge(df_merged, df4, on=[\'start_time\',\'TAC_AREA_NAME\'], how=\'inner\')\n'

In [20]:
df_wide = df_lmp.pivot_table(index=['start_time','NODE_ID'], columns='LMP_TYPE', values='MW', aggfunc='first')

df_wide['LMP_recalc'] = (
    df_wide[['MCE','MCC','MCL','MGHG']]
    .fillna(0)
    .sum(axis=1)
)

df_wide['Check'] = df_wide['LMP_recalc'] - df_wide['LMP']

print(df_wide)

LMP_TYPE                                             LMP      MCC       MCE  \
start_time                NODE_ID                                             
2025-12-10 08:00:00+00:00 AFPR_1_TOT_GEN-APND   45.18631  0.00000  47.36511   
                          AFTON_3_AFTONCC-APND  44.86897  0.00000  47.36511   
                          AGRICO_6_PL3N5-APND   48.55397  0.00000  47.36511   
                          AGRICO_7_UNIT-APND    48.55397  0.00000  47.36511   
                          ALAMIT_2_PL1X3-APND   47.14249  0.00000  47.36511   
...                                                  ...      ...       ...   
2025-12-11 07:00:00+00:00 PGF1_2_PDRP52-APND    46.92580 -0.43068  45.27022   
                          PGF1_2_PDRP53-APND    46.92580 -0.43068  45.27022   
                          PGF1_2_PDRP54-APND    46.92580 -0.43068  45.27022   
                          PGF1_2_PDRP55-APND    46.92580 -0.43068  45.27022   
                          PGF1_2_PDRP56-APND    46.9